<a href="https://colab.research.google.com/github/leadministrador/mineria-datos-ii-proyecto-integrador/blob/main/notebooks/02_streaming_bronze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pyspark

In [ ]:
!unzip -q -o cloud_provider_challenge_dataset_v1.zip -d /content/proyecto

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("proyecto-integrador")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Versión de Spark: 4.0.5
Master: local[*]
Shuffle partitions: 8


In [ ]:
LANDING = "/content/proyecto/datalake/landing"
eventos = spark.read.json(LANDING + "/usage_events_stream")
eventos.printSchema()
print("Eventos:", eventos.count())

root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)

Eventos: 43200


In [ ]:
esquema = ("event_id STRING, timestamp STRING, org_id STRING, resource_id STRING, "
           "service STRING, region STRING, metric STRING, value STRING, unit STRING, "
           "cost_usd_increment DECIMAL(10,4), schema_version BIGINT, "
           "carbon_kg DECIMAL(10,4), genai_tokens BIGINT")
print("Esquema listo")

Esquema listo


In [ ]:
BRONZE = "/content/proyecto/datalake/bronze/usage_events"
CHECK = "/content/proyecto/checkpoints/bronze_usage_events"

def ingesta():
    eventos_stream = (spark.readStream
        .schema(esquema)
        .option("maxFilesPerTrigger", 10)
        .json(LANDING + "/usage_events_stream")
        .withColumn("event_ts", F.col("timestamp").cast("timestamp"))
        .withColumn("event_date", F.to_date("event_ts"))
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withWatermark("event_ts", "60 days")
        .dropDuplicatesWithinWatermark(["event_id"]))
    q = (eventos_stream.writeStream
        .format("parquet")
        .option("path", BRONZE)
        .option("checkpointLocation", CHECK)
        .partitionBy("event_date")
        .outputMode("append")
        .trigger(availableNow=True)
        .start())
    q.awaitTermination()

ingesta()
print("Filas en Bronze:", spark.read.parquet(BRONZE).count())

Filas en Bronze: 43200


In [ ]:
antes = spark.read.parquet(BRONZE).count()
ingesta()
despues = spark.read.parquet(BRONZE).count()
print("Antes:", antes, "· Después:", despues)

spark.read.parquet(BRONZE).select("event_id", "event_date", "source_file", "ingest_ts").show(3, truncate=False)

Antes: 43200 · Después: 43200
+----------------+----------+----------------------+-----------------------+
|event_id        |event_date|source_file           |ingest_ts              |
+----------------+----------+----------------------+-----------------------+
|evt_95brfj7kohgq|2025-08-26|events_part_0119.jsonl|2026-11-05 01:17:33.661|
|evt_cpftlo5wx4vm|2025-08-26|events_part_0119.jsonl|2026-11-05 01:17:33.661|
|evt_powjl26st343|2025-08-26|events_part_0119.jsonl|2026-11-05 01:17:33.661|
+----------------+----------+----------------------+-----------------------+
only showing top 3 rows


In [ ]:
SILVER = "/content/proyecto/datalake/silver/usage_events"
QUARANTINE = "/content/proyecto/datalake/quarantine/usage_events"

esquema_bronze = esquema + ", event_ts TIMESTAMP, event_date DATE, ingest_ts TIMESTAMP, source_file STRING"

RULE = ("coalesce(event_id IS NOT NULL AND cost_usd_increment >= -0.01 "
        "AND (value IS NULL OR unit IS NOT NULL), false)")

orgs = (spark.read.option("header", True).option("inferSchema", True)
        .csv(LANDING + "/customers_orgs.csv")
        .select("org_id", "industry", "plan_tier"))

bronze_stream = spark.readStream.schema(esquema_bronze).parquet(BRONZE)

silver = (bronze_stream.filter(RULE)
          .withColumn("value", F.col("value").cast("decimal(12,4)"))
          .join(F.broadcast(orgs), "org_id", "left"))
quarantine = bronze_stream.filter(f"NOT {RULE}")

(silver.writeStream.format("parquet")
    .option("path", SILVER)
    .option("checkpointLocation", "/content/proyecto/checkpoints/silver_usage_events")
    .partitionBy("event_date")
    .trigger(availableNow=True).start().awaitTermination())

(quarantine.writeStream.format("parquet")
    .option("path", QUARANTINE)
    .option("checkpointLocation", "/content/proyecto/checkpoints/quarantine_usage_events")
    .trigger(availableNow=True).start().awaitTermination())

print("Silver:", spark.read.parquet(SILVER).count())
print("Quarantine:", spark.read.parquet(QUARANTINE).count())

Silver: 40956
Quarantine: 2244


In [ ]:
for tabla in ["customers_orgs", "users", "billing_monthly"]:
    df = (spark.read.option("header", True).option("inferSchema", True)
          .csv(LANDING + "/" + tabla + ".csv")
          .withColumn("ingest_ts", F.current_timestamp())
          .withColumn("source_file", F.col("_metadata.file_name")))
    df.write.mode("overwrite").parquet("/content/proyecto/datalake/bronze/" + tabla)
    print(tabla, spark.read.parquet("/content/proyecto/datalake/bronze/" + tabla).count())

customers_orgs 80
users 800
billing_monthly 240


In [ ]:
def leer(tabla, esquema_m, clave):
    return (spark.read.option("header", True).schema(esquema_m)
            .csv(LANDING + "/" + tabla + ".csv")
            .dropDuplicates([clave])
            .withColumn("ingest_ts", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_name")))

BRONZE_M = "/content/proyecto/datalake/bronze/"

In [ ]:
leer("customers_orgs",
     "org_id STRING, org_name STRING, industry STRING, hq_region STRING, "
     "plan_tier STRING, is_enterprise STRING, signup_date DATE, sales_rep STRING, "
     "lifecycle_stage STRING, marketing_source STRING, nps_score DECIMAL(5,1)",
     "org_id").write.mode("overwrite").parquet(BRONZE_M + "customers_orgs")

print("customers_orgs", spark.read.parquet(BRONZE_M + "customers_orgs").count())

customers_orgs 80


In [ ]:
leer("users",
     "user_id STRING, org_id STRING, email STRING, role STRING, active STRING, "
     "created_at DATE, last_login DATE",
     "user_id").write.mode("overwrite").parquet(BRONZE_M + "users")

print("users", spark.read.parquet(BRONZE_M + "users").count())

users 800


In [ ]:
leer("billing_monthly",
     "invoice_id STRING, org_id STRING, month DATE, subtotal DECIMAL(12,2), "
     "credits DECIMAL(12,2), taxes DECIMAL(12,2), currency STRING, "
     "exchange_rate_to_usd DECIMAL(10,5)",
     "invoice_id").write.mode("overwrite").partitionBy("month").parquet(BRONZE_M + "billing_monthly")

print("billing_monthly", spark.read.parquet(BRONZE_M + "billing_monthly").count())

billing_monthly 240


In [ ]:
SILVER_B = "/content/proyecto/datalake/silver/billing_monthly"

silver_billing = (spark.read.parquet(BRONZE_M + "billing_monthly")
    .withColumn("credits", F.expr("coalesce(credits, 0)")))

silver_billing.write.mode("overwrite").partitionBy("month").parquet(SILVER_B)

print("Silver billing:", spark.read.parquet(SILVER_B).count())

Silver billing: 240


In [ ]:
GOLD = "/content/proyecto/datalake/gold/org_daily_usage_by_service"
silver_ev = spark.read.parquet(SILVER)
claves = ["org_id", F.col("event_date").alias("usage_date"), "service"]

costos = (silver_ev.groupBy(claves)
          .agg(F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
               F.sum("genai_tokens").alias("genai_tokens"),
               F.round(F.sum("carbon_kg"), 4).alias("carbon_kg")))

requests = (silver_ev.filter("metric = 'requests'")
            .groupBy(claves)
            .agg(F.sum("value").alias("requests")))

gold = costos.join(requests, ["org_id", "usage_date", "service"], "left")
gold.write.mode("overwrite").partitionBy("usage_date").parquet(GOLD)

print("Gold filas:", spark.read.parquet(GOLD).count())
spark.read.parquet(GOLD).show(5)

Gold filas: 11050
+------------+----------+--------------+------------+---------+--------+----------+
|      org_id|   service|daily_cost_usd|genai_tokens|carbon_kg|requests|usage_date|
+------------+----------+--------------+------------+---------+--------+----------+
|org_he7ur23g|networking|        2.8393|        NULL|   0.0528|260.0000|2025-08-12|
|org_9t84azyt|   storage|        1.9220|        NULL|   0.0233|116.0000|2025-08-12|
|org_jxepq85j|   storage|        0.1158|        NULL|   0.0013|    NULL|2025-08-12|
|org_teiyzcot|networking|        2.5873|        NULL|   0.0542|241.0000|2025-08-12|
|org_9t84azyt|  database|       13.4262|        NULL|   0.0555|273.0000|2025-08-12|
+------------+----------+--------------+------------+---------+--------+----------+
only showing top 5 rows


In [ ]:
!rm -rf /content/proyecto/datalake/quarantine/usage_events /content/proyecto/checkpoints/quarantine_usage_events

In [ ]:
quarantine = (bronze_stream.filter(f"NOT {RULE}")
    .withColumn("motivo",
        F.when(F.col("event_id").isNull(), "sin_event_id")
         .when(F.col("cost_usd_increment") < -0.01, "costo_negativo")
         .otherwise("unit_vacio")))

q = (quarantine.writeStream.queryName("quarantine_usage_events")
    .format("parquet")
    .option("path", QUARANTINE)
    .option("checkpointLocation", "/content/proyecto/checkpoints/quarantine_usage_events")
    .trigger(availableNow=True).start())
q.awaitTermination()

print(q.name, q.lastProgress["numInputRows"])
spark.read.parquet(QUARANTINE).groupBy("motivo").count().show()

quarantine_usage_events 43200
+--------------+-----+
|        motivo|count|
+--------------+-----+
|costo_negativo|  211|
|    unit_vacio| 2033|
+--------------+-----+



In [22]:
spark.read.parquet(QUARANTINE).select("event_id", "cost_usd_increment", "value", "unit", "motivo").show(6, truncate=False)

+----------------+------------------+------+----+----------+
|event_id        |cost_usd_increment|value |unit|motivo    |
+----------------+------------------+------+----+----------+
|evt_ls602epdij1k|0.0612            |3.0463|NULL|unit_vacio|
|evt_4u08mhaeqytm|0.0000            |0.0   |NULL|unit_vacio|
|evt_z2csvu281t4y|1.7901            |121.0 |NULL|unit_vacio|
|evt_f38p2f6ehxfl|1.7223            |131.0 |NULL|unit_vacio|
|evt_okc2gs3di1yv|0.1568            |3.3228|NULL|unit_vacio|
|evt_bfbjcfovlg7u|0.1076            |1.333 |NULL|unit_vacio|
+----------------+------------------+------+----+----------+
only showing top 6 rows


In [23]:
spark.read.parquet(QUARANTINE).filter("motivo = 'costo_negativo'").select("event_id", "cost_usd_increment", "value", "unit", "motivo").show(3, truncate=False)

+----------------+------------------+-------+--------+--------------+
|event_id        |cost_usd_increment|value  |unit    |motivo        |
+----------------+------------------+-------+--------+--------------+
|evt_pmlgsgb29mur|-7.9514           |129.0  |count   |costo_negativo|
|evt_9y3nrmfky3lu|-0.8771           |10.5777|gb_hours|costo_negativo|
|evt_fac9pru3xlpd|-0.5116           |10.3166|gb_hours|costo_negativo|
+----------------+------------------+-------+--------+--------------+
only showing top 3 rows
